# Exercice 4 : À quoi ressemble le « normal » ? (MMD-critic)

> **En une phrase :** on résume chaque classe par quelques exemples typiques, pour voir où le normal et l'attaque **se ressemblent**.

**Deux mots de vocabulaire.** MMD-critic (Kim et al., 2016) choisit dans un jeu de données :
- des **prototypes** : les quelques flux qui résument le mieux le groupe (« le flux normal typique ») ;
- des **criticismes** : les flux que les prototypes représentent le plus mal (les cas atypiques).

**Pourquoi c'est utile.** Pour l'attaquant, les prototypes du trafic normal sont **la cible à imiter**. Pour le défenseur, les criticismes montrent **les cas rares** du jeu de données.

**Le plan.**
1. Préparer l'IDS.
2. Trouver les prototypes et criticismes du trafic normal.
3. Comparer une attaque détectée à ces prototypes.
4. Mesurer et visualiser le chevauchement entre normal et attaques.
5. Les attaques atypiques sont-elles bien détectées ?

## Étape 0 : exécutez cette cellule en premier
Sur Colab (notebook ouvert depuis Drive), elle télécharge `helper.py` et les données. Elle doit afficher **Prêt : True**.

In [ ]:
# Récupère helper.py et les données depuis GitHub s'ils ne sont pas à côté du notebook
# (cas d'un notebook ouvert depuis Google Drive). En local, cette cellule ne fait rien.
import os, sys
if not os.path.exists('helper.py'):
    !test -d /content/cours || git clone -q --depth 1 https://github.com/elyesmanai/8IAR805---Cybersecurity-AI.git /content/cours
    %cd "/content/cours/Chapitre 5 - Sécurité de l'IA"
print('Prêt :', os.path.exists('helper.py'))

## Étape 1 : Préparer l'IDS

In [1]:
from helper import preparer
X_train, y_train, X_test, y_test, types_attaque, ids = preparer()

IDS de l'entreprise -> accuracy = 0.901 | recall (attaques détectées) = 0.864


**👀 Ce qu'on doit voir :** accuracy ≈ 0,90 et recall ≈ 0,86 : l'IDS détecte 86 % des attaques. C'est notre point de départ.

## Étape 2 : Prototypes et criticismes du trafic normal
**Ce qu'on fait :** on prend 1 500 flux normaux et on demande 5 prototypes et 5 criticismes. On affiche 10 colonnes lisibles.

In [2]:
from helper import mmd_critic

normal = X_test[y_test == 0].sample(1500, random_state=0)   # garder <= 2000 : le calcul est en n²

# <-- MODIFIABLE : le nombre de prototypes et de criticismes
P, C = mmd_critic(normal.values, n_proto=5, n_crit=5)

colonnes = ['proto', 'service', 'state', 'dur', 'sbytes', 'dbytes', 'spkts', 'dpkts', 'sttl', 'rate']
print('PROTOTYPES : le flux normal typique')
display(normal.iloc[P][colonnes])
print('CRITICISMES : les flux normaux atypiques')
display(normal.iloc[C][colonnes])

PROTOTYPES : le flux normal typique


,proto,service,state,dur,sbytes,dbytes,spkts,dpkts,sttl,rate
14313,111,0,3,0.005972,1470,1728,22,14,31,5860.683346
17791,117,2,2,0.000987,146,178,2,2,31,3039.513836
42790,111,0,3,0.587552,644,268,10,6,254,25.529655
12662,111,0,3,0.038905,5174,83812,90,92,31,4652.358494
30633,111,5,3,1.004612,1580,10168,12,18,31,28.866867


CRITICISMES : les flux normaux atypiques


,proto,service,state,dur,sbytes,dbytes,spkts,dpkts,sttl,rate
25002,111,5,3,1.045973,1580,10168,12,18,31,27.725383
26499,111,5,3,1.178145,1684,10168,14,18,31,26.312550
19351,111,5,3,1.012026,1684,10168,14,18,31,30.631626
26366,111,5,3,1.060084,1684,10168,14,18,31,29.242966
2641,111,5,3,1.012866,1684,10168,14,18,31,30.606220


**👀 Ce qu'on doit voir :** - **prototypes** : des flux variés (petites requêtes de 146 octets, échanges plus gros), presque tous avec `sttl = 31`. C'est le TTL du trafic normal.
- **criticismes** : 5 flux presque identiques (service 5, ≈ 1 600 octets, ≈ 1 s). Un petit groupe de flux qui se répète et qu'aucun prototype ne représente bien.

**❓ Question pour la classe :** si vous étiez l'attaquant, quelles valeurs viseriez-vous pour `sbytes`, `dur` et `sttl` ?

## Étape 3 : Comparer une attaque détectée aux prototypes
**Ce qu'on fait :** on met une attaque détectée par l'IDS à côté des 5 prototypes normaux.

In [3]:
import pandas as pd

attaque = X_test[(y_test == 1) & (ids.predict(X_test) == 1)].sample(1, random_state=3)
comparaison = pd.concat([attaque[colonnes], normal.iloc[P][colonnes]])
comparaison.index = ['ATTAQUE'] + [f'prototype {i+1}' for i in range(len(P))]
comparaison

,proto,service,state,dur,sbytes,dbytes,spkts,dpkts,sttl,rate
ATTAQUE,111,7,3,4.068827,1562,112450,34,102,62,33.179094
prototype 1,111,0,3,0.005972,1470,1728,22,14,31,5860.683346
prototype 2,117,2,2,0.000987,146,178,2,2,31,3039.513836
prototype 3,111,0,3,0.587552,644,268,10,6,254,25.529655
prototype 4,111,0,3,0.038905,5174,83812,90,92,31,4652.358494
prototype 5,111,5,3,1.004612,1580,10168,12,18,31,28.866867


**👀 Ce qu'on doit voir :** l'attaque se distingue surtout par `sttl = 62` (contre 31 pour presque tous les prototypes) et par un très gros `dbytes` (≈ 112 000).

**❓ Question pour la classe :** sur quelles features l'attaque est-elle la plus éloignée des prototypes ? L'attaquant contrôle-t-il ces features ?

## Étape 4 : Le normal et l'attaque se chevauchent-ils ?
**Ce qu'on fait :**
1. On met les deux classes dans le même espace (même normalisation).
2. On calcule 10 prototypes pour chaque classe.
3. On rattache chaque flux à son prototype le plus proche.

Si une attaque est rattachée à un prototype **normal**, elle ressemble déjà à du trafic normal : c'est une candidate idéale pour l'évasion.

In [4]:
from sklearn.preprocessing import StandardScaler
from helper import taux_chevauchement

attaques = X_test[y_test == 1].sample(1500, random_state=0)
normalisation = StandardScaler().fit(pd.concat([normal, attaques]))
Z_normal, Z_attaque = normalisation.transform(normal), normalisation.transform(attaques)

# <-- MODIFIABLE : le nombre de prototypes par classe
P_n, C_n = mmd_critic(Z_normal, n_proto=10, n_crit=10, deja_normalise=True)
P_a, C_a = mmd_critic(Z_attaque, n_proto=10, n_crit=10, deja_normalise=True)

attaques_chez_normal, normaux_chez_attaque = taux_chevauchement(Z_normal, Z_attaque, P_n, P_a)
print(f"Attaques qui ressemblent le plus à un prototype NORMAL : {attaques_chez_normal:.0%}")
print(f"Flux normaux qui ressemblent le plus à un prototype ATTAQUE : {normaux_chez_attaque:.0%}")

Attaques qui ressemblent le plus à un prototype NORMAL : 24%
Flux normaux qui ressemblent le plus à un prototype ATTAQUE : 16%


**👀 Ce qu'on doit voir :** ≈ 24 % des attaques ressemblent davantage à du trafic normal qu'à une attaque typique. Dans l'autre sens, ≈ 16 % des flux normaux ressemblent à des attaques.

**Visualisation 3D.** On réduit les 42 features à 3 axes (ACP) pour pouvoir regarder. **Faites tourner le graphique avec la souris.**

In [5]:
from helper import nuage_3d
nuage_3d(Z_normal, Z_attaque, P_n, C_n, P_a, C_a)

**👀 Ce qu'on doit voir :** bleu = normal, orange = attaques, losanges = prototypes, croix = criticismes. On voit des zones où les deux couleurs se mélangent.

⚠️ La 3D ne garde qu'une partie de l'information (le % est dans le titre). Deux points proches à l'écran peuvent être loin en réalité : le **chiffre** de la cellule précédente fait foi.

In [6]:
# Quels types d'attaques ressemblent le plus au normal ?
from helper import types_proches_du_normal
types_proches_du_normal(Z_normal, Z_attaque, P_n, P_a, types_attaque.loc[attaques.index])

Shellcode         0.79
Analysis          0.57
DoS               0.43
Reconnaissance    0.40
Fuzzers           0.39
Backdoor          0.35
Exploits          0.27
Generic           0.02
Worms             0.00
Name: part qui ressemble au normal, dtype: float64

**👀 Ce qu'on doit voir :** **Shellcode** (≈ 80 %) et **Analysis** (≈ 57 %) ressemblent le plus au normal, puis DoS, Reconnaissance et Fuzzers (≈ 40 %). Les attaques **Generic** presque jamais (2 %).

**❓ Question pour la classe :** ces types ressemblent-ils à ceux qu'on a trouvés dans la zone grise à l'ex. 1 ?

## Étape 5 : Les attaques atypiques sont-elles bien détectées ?
**Ce qu'on fait :** on prend les 100 criticismes des attaques (les plus atypiques) et on regarde si l'IDS les détecte aussi bien que les autres.

In [7]:
P_att, C_att = mmd_critic(attaques.values, n_proto=20, n_crit=100)
detecte = ids.predict(attaques)

print(f"Détection sur toutes les attaques : {detecte.mean():.0%}")
print(f"Détection sur les prototypes      : {detecte[P_att].mean():.0%}")
print(f"Détection sur les criticismes     : {detecte[C_att].mean():.0%}")
types_attaque.loc[attaques.index[C_att]].value_counts()

Détection sur toutes les attaques : 87%
Détection sur les prototypes      : 85%
Détection sur les criticismes     : 100%


attack_cat
Generic     65
DoS         16
Exploits    13
Backdoor     4
Fuzzers      2
Name: count, dtype: int64

**👀 Ce qu'on doit voir :** surprise : les criticismes sont détectés à **100 %**, mieux que la moyenne (≈ 87 %). Ici, les attaques atypiques sont surtout des **Generic** (65 sur 100), une famille que l'IDS reconnaît très bien. Atypique ne veut pas dire invisible.

**❓ Question pour la classe :** qu'est-ce qui est le plus dangereux pour l'IDS : une attaque atypique, ou une attaque qui ressemble à un prototype normal (étape 4) ?

## Ce qu'il faut retenir
- Les prototypes du trafic normal donnent à l'attaquant **une cible concrète à imiter** (ex : `sttl = 31`).
- Le danger, ce ne sont pas les attaques bizarres : ce sont celles qui **ressemblent déjà au normal** (≈ 24 % ici).
- Pour le défenseur, les criticismes et le chevauchement montrent où collecter plus de données, ou quels cas envoyer à un analyste.

**Exercice suivant :** l'attaquant a tout ce qu'il faut (une copie, quoi modifier, vers quoi aller). Il passe à l'attaque.